In [0]:
import json 
import os 
from urllib.parse import urljoin

import pandas as pd 
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/' 
FILES_URL = 'https://cvijan.github.io/nss-api/files/' 
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True) 
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

# get_json(BASE_URL + 'orders/page-99.json')
get_json(BASE_URL + 'status.json')['service']


'Northern Star Shop API'

In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    
    return records

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

orders = fetch_all(BASE_URL + 'orders/page-1.json')

def save_json(records: list, path: str) -> None: 
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
# 4. Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити у
# задатку 5

def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes unchanged
    return len(response.content)

# За разлику од get_json, ова функција не чита садржај, већ само копира фајл на диск, бајт по
# бајт
# Читање долази после, са pd.read_csv и pd.read_parquet
# Parquet није обичан текст, па се уписује са 'wb', а не са 'w'
# Проба: download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,
# 'teams.csv'))
# Повратна вредност: 801, а у data/raw се појављује фајл teams.csv

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams',
# 'matches'
# 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
# 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције
# download_file
# 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
# Након тога преименовати колоне
# customer_name у customer
# customer_city у city
# 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
# По потреби, pd.read_csv(...) или pd.read_parquet(...)
# 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)

def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR, 'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR, 'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR, 'matches.parquet'))

    orders = pd.json_normalize(orders, sep='_').rename(columns={
        'customer_name': 'customer',
        'customer_city': 'city'
    })

    products = pd.read_csv(os.path.join(raw_dir, 'products.csv'))
    teams = pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    matches = pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))

    # products = pd.read_csv(FILES_URL + 'products.csv')
    # teams = pd.read_csv(FILES_URL + 'teams.csv')
    # matches = pd.read_parquet(FILES_URL + 'matches.parquet')

    return {
        'orders': orders,
        'products': products,
        'teams': teams,
        'matches': matches
    }

# orders = extract(RAW_DIR)['orders']
# orders

In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame
# која чисти каталог производа
# Ако се понавља исти производ, садржати први ред за сваки product_id
# team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред
# остаје празан (то су производи лиге, без тима)
# category треба да буде без размака, има мала слова
# price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане
# само float (у сваком случају)
# Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
# active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало
# постаје False
# Output: 105 редова, price је типа float64, active је True за 101 производ
# Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves',
# 'price': 1490.0, 'active': True, ...}

def clean_products(df: pd.DataFrame) -> pd.DataFrame: 
    clean = df.copy()
    clean = clean[~clean.duplicated(subset='product_id', keep='first')]

    clean['team_code'] = clean['team_code'].str.strip().str.upper()
    clean['category'] = clean['category'].str.strip().str.lower()

    clean['price'] = clean['price'].astype(str).str.replace('RSD', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False).astype(float)

    if clean['active'].dtype != bool:
        clean['active'] = clean['active'].str.strip().str.lower().eq('yes')

    return clean

In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame
# order_id, customer, city, payment без размака;
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()

    clean['city'] = clean['city'].fillna('Unknown')

    for col in ['order_id', 'customer', 'city', 'payment']:
        clean[col] = clean[col].astype(str).str.strip()
    
    clean['order_id'] = clean['order_id'].astype(str).str.upper()

    for col in ['customer', 'city']:
        clean[col] = clean[col].astype(str).str.title()
        
    return clean

In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    df = df.copy()
    duplicates = df[df.duplicated(subset='order_id', keep='first')]
    uniques = df.drop_duplicates(subset='order_id', keep='first')

    return uniques, duplicates

In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа
# 'OK' за исправан ред, у супротном разлог
# Провере радити редом и вратити први пронађени проблем
# order_id не почиње са NSS- -> 'invalid_order'
# quantity недостаје -> 'missing_quantity'
# quantity није цео број већи од нуле -> 'invalid_quantity'
# unit_price није број већи од нуле -> 'invalid_price'
# product_id не постоји међу производима -> 'unknown_product'
# Помоћ:
# pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
# pd.isna(value) препознаје и None и NaN

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row.get('quantity'), errors='coerce')
    unit_price = pd.to_numeric(row.get('unit_price'), errors='coerce')

    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    elif pd.isna(quantity):
        return 'missing_quantity'
    elif quantity <= 0 or quantity % 1 != 0:
        return 'invalid_quantity'
    elif pd.isna(unit_price) or unit_price <= 0:
        return 'invalid_price'
    elif not row['product_id'] in product_ids:
        return 'unknown_product'
    
    return 'OK'

In [0]:
# 10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# pd.DataFrame
# Типови:
# quantity -> int
# unit_price -> float
# order_date -> датум (pd.to_datetime)
# Нова колона total (quantity * unit_price)
# Из очишћених производа (products) додати product_name, team_code и category
# (merge по product_id)
# Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = df['quantity'].astype(int)
    df['unit_price'] = df['unit_price'].astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']

    cleaned_products = clean_products(products)

    # print(cleaned_products)

    df = pd.merge(
        df,
        cleaned_products[['product_id', 'product_name', 'team_code', 'category']],
        on='product_id',
        how='left'
    )

    return df

In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,
# rejected)
# valid су трансформисани исправни редови
# rejected су очишћени неисправни редови и дупликати, са колоном reason
# (дупликати за разлог треба да имају 'duplicate')
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

# orders = orders.to_dict(orient='records')
# print(orders)

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = df.copy()
    products = products.copy()

    cleaned_products = clean_products(products)
    df = clean_orders(df)

    product_ids = set(cleaned_products['product_id'].unique())

    valid_orders = []
    invalid_orders = []

    valid, rejected = remove_duplicate_orders(df)
    for order in valid.to_dict(orient='records'):
        result = validate_order(order, product_ids)

        if result == 'OK':
            valid_orders.append(order)
        else:
            invalid_orders.append({
                **order,
                'reason': result
            })

    rejected = rejected.copy()
    rejected['reason'] = 'duplicate'

    invalid_orders.extend(rejected.to_dict(orient='records'))
    valid_df = pd.DataFrame(valid_orders)
    valid_df = transform_orders(valid_df, products)

    
    return valid_df, pd.DataFrame(invalid_orders)

In [0]:
# 12. Резултати тимова. Написати функцију team_results(matches: pd.DataFrame, teams:
# pd.DataFrame) -> pd.DataFrame која сваку одиграну утакмицу претвара у два реда, један
# за домаћина и један за госта.
# Користити само утакмице са status == 'finished' (одложена нема резултат)
# Колоне:
# match_id
# match_date (Датум из kickoff_ts, без времена -
# df['kickoff_ts'].dt.normalize())
# team_code, opponent_code
# goals_for, goals_against
# result ('W', 'D', 'L'), points (3, 1, 0)
# Утакмице имају team_id, а производи team_code

def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:

    df = matches[matches['status'] == 'finished'].copy()
    df['match_date'] = df['kickoff_ts'].dt.normalize()

    tm = teams.copy()

    home_teams = tm.rename(columns={'team_id': 'home_team_id', 'code': 'team_code'})

    away_teams = tm.rename(columns={'team_id': 'away_team_id', 'code': 'opponent_code'})

    merged = pd.merge(
        df,
        home_teams,
        on='home_team_id',
        how='left'
    )

    merged = merged.merge(
        away_teams,
        on='away_team_id',
        how='left'
    )

    home = merged[
        ['match_id', 'match_date', 'team_code', 'opponent_code','home_goals', 'away_goals']
    ].copy()

    home = home.rename(columns={'home_goals': 'goals_for', 'away_goals': 'goals_against'})

    away = merged[
        ['match_id', 'match_date', 'opponent_code', 'team_code', 'home_goals', 'away_goals']
    ].copy()

    away = away.rename(columns={
        'opponent_code': 'team_code', 'team_code': 'opponent_code', 
        'home_goals': 'goals_against', 'away_goals': 'goals_for'
    })

    m = pd.concat([home, away], ignore_index=True)

    wins = m['goals_for'] > m['goals_against']
    draws = m['goals_for'] == m['goals_against']

    m.loc[wins, 'result'] = 'W'
    m.loc[draws, 'result'] = 'D'
    m.loc[~(wins | draws), 'result'] = 'L'

    m.loc[m['result'] == 'W', 'points'] = 3
    m.loc[m['result'] == 'D', 'points'] = 1
    m.loc[m['result'] == 'L', 'points'] = 0

    return m

In [0]:
# 13. Табела лиге. Написати функцију league_table(results: pd.DataFrame) -> pd.DataFrame
# која од резултата из задатка 12 прави табелу
# Колоне:
# position, team_code
# played, won, drawn, lost
# goals_for, goals_against, goal_diff
# points
# Редослед табеле: Бодови, па гол разлика, па дати голови (све опадајуће)
# Output: 20 тимова, први FWR (26 бодова), други BMC (23)
# HGA и OKT имају по 11 утакмица

# results = team_results(matches, teams)
# print(results)

def league_table(results: pd.DataFrame) -> pd.DataFrame:
    res = results.copy()

    res['won'] = res['result'] == 'W'
    res['drawn'] = res['result'] == 'D'
    res['lost'] = res['result'] == 'L'

    res = res.groupby('team_code').agg(
        played=('match_id', 'count'),
        won=('won', 'sum'),
        drawn=('drawn', 'sum'),
        lost=('lost', 'sum'),
        goals_for=('goals_for', 'sum'),
        goals_against=('goals_against', 'sum'),
        points=('points', 'sum')
    ).reset_index()

    res['goal_diff'] = res['goals_for'] - res['goals_against']

    res = res.sort_values(
        by=['points', 'goal_diff', 'goals_for'],
        ascending=False
    ).reset_index(drop=True)

    res.insert(0, 'position', range(1, len(res) + 1))

    return res

In [0]:
# 14. Прозори после утакмица. Написати функцију match_windows(results: pd.DataFrame) ->
# pd.DataFrame која за сваки ред из задатка 12 прави 4 реда: дан утакмице и три дана после
# ње, са резултатом те утакмице
# Зашто: да бисмо поредили продају "у данима после победе" и "у данима после
# пораза", сваки дан мора да зна да ли је у прозору од 3 дана после утакмице свог тима
# и какав је био резултат.
# Пример: BRU је 2026-08-15 победио, па један ред из задатка 12 (team_results)
# постаје:

# team_code   date          window
# BRU         2026-08-15    W
# BRU         2026-08-16    W
# BRU         2026-08-17    W
# BRU         2026-08-18    W

# Помоћ:
# За сваки помак d од 0 до 3 (for d in range(4)) направити копију колона
# team_code, match_date, result и додати колону date померену за d дана
# Све 4 копије скупити у листу и спојити са pd.concat(..., ignore_index=True)
# Задржати колоне team_code, date, result и преименовати result у window
# Тимови играју најчешће једном у 6 дана, па се прозори једног тима никада не
# преклапају
# Output: 952 реда (238 * 4)
# колоне team_code, date, window
# четири реда за BRU од 2026-08-15 до 2026-08-18 су као у примеру изнад

def match_windows(results: pd.DataFrame) -> pd.DataFrame:
    df = results.copy()
    windows = []

    for d in range(4):
        window = df[['team_code', 'match_date', 'result']].copy()
        window['date'] = window['match_date'] + pd.Timedelta(days=d)

        windows.append(window)

    df = pd.concat(windows, ignore_index=True)

    df = df[['team_code', 'date', 'result']].rename(
        columns={'result': 'window'}
    )

    return df

In [0]:
# 15. Промет по колони. Написати функцију revenue_by(df: pd.DataFrame, column: str) ->
# pd.DataFrame
# Kолоне [column, 'revenue'], сортирано опадајуће
# Input: valid, 'team_code' -> Output: 20 редова, први LSB 720420.00, последњи WFT
# 234650.00
# Input:valid, 'category' → Output: jerseys 7094410.00 на првом месту

def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df = df.copy()

    return df.groupby(column).agg(
        revenue=('total', 'sum')
    ).reset_index().sort_values(by='revenue', ascending=False).reset_index(drop=True)

In [0]:
# 16. Продаја после резултата. Написати функцију sales_after_results(valid: pd.DataFrame,
# windows: pd.DataFrame) -> pd.DataFrame која одговара на питање пројекта
# 1. Календар: Сваки тим x Сваки дан.
# 2. Дневни промет по тиму и дану
# valid.groupby(['team_code', 'order_date'])
# ['total'].sum().reset_index(name='revenue')
# Спојити га на календар (дан без продаје добија NaN за revenue, замените то са
# 0)
# 3. Спојити календар са прозорима добијених из match_windows
# calendar.merge(windows, left_on=['team_code', 'order_date'],
# right_on=['team_code', 'date'], how='left')
# 4. Дан који није ни у једном прозору нема резултат и добија 'none'
# calendar['window'] = calendar['window'].fillna('none')
# 5. Груписати по window: team_days (број дана), revenue (збир), avg_daily_revenue
# (просек, 2 децимале), сортирано опадајуће

def sales_after_results(valid: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    df = valid.copy()
    windows = windows.copy()

    # 1. Create calendar: every team x every day
    calendar_teams = pd.DataFrame({'team_code': windows['team_code'].unique()})

    days = pd.DataFrame({'order_date': df['order_date'].unique()})

    calendar = calendar_teams.merge(days, how='cross')

    # 2. Calculate daily revenue by team
    daily_revenue = (
        df.groupby(['team_code', 'order_date']).agg(revenue=('total', 'sum')).reset_index()
    )

    # 3. Add daily revenue to calendar
    calendar = calendar.merge(
        daily_revenue,
        on=['team_code', 'order_date'],
        how='left'
    )

    calendar['revenue'] = calendar['revenue'].fillna(0)

    # 4. Add match windows
    calendar = calendar.merge(
        windows,
        left_on=['team_code', 'order_date'],
        right_on=['team_code', 'date'],
        how='left'
    )

    # 5. Days outside match windows get 'none'
    calendar['window'] = calendar['window'].fillna('none')

    # 6. Aggregate by match result
    result = (
        calendar.groupby('window').agg(
            team_days=('team_code', 'count'),
            revenue=('revenue', 'sum'),
            avg_daily_revenue=('revenue', 'mean')
        ).reset_index()
    )

    result['avg_daily_revenue'] = result['avg_daily_revenue'].round(2)

    result = result.sort_values(
        'revenue',
        ascending=False
    ).reset_index(drop=True)

    return result

In [0]:
# 17. Load. Написати функцију load(outputs: dict, processed_dir: str) -> dict која сваки
# DataFrame из речника {име_фајла: DataFrame} уписује у processed_dir и враћа
# {име_фајла: број_редова}

def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
        written[name] = len(df)
    return written

In [0]:
# 18. Написати функцију run_pipeline(raw_dir: str, processed_dir: str) -> dict која
# покреће цео pipeline

def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=
['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric', 
'value']).to_csv(
        os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary

run_pipeline(RAW_DIR, PROCESSED_DIR)

{'total_rows': 1852,
 'duplicates': 12,
 'valid_rows': 1800,
 'rejected_rows': 52,
 'total_revenue': 10045400.0,
 'files_written': 8}